# Quadrant Results Table Consolidation

Consolidates regional performance metrics (`SP`, `PF`, `PD` with uncertainties, as percentages $\times 100$) from `quadrant_results.csv` generated by `scripts/run_quadrant_analysis.py` into a standardized LaTeX-formatted grid table (grouped by $E_T$ energy intervals with `SP`, `PF`, and `PD` sub-columns across $|\eta|$ bins) and exports to CSV.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.core.Validation.QuadrantTableConsolidator import QuadrantTableConsolidator

print(f"Project root resolved: {PROJECT_ROOT}")

### 1. Configuration & Strategy Selection

Select the strategy (`strategy1` or `strategy2`), input CSV file, and formatting options (percentage scaling, LaTeX math delimiters, precision).

In [ ]:
# Strategy selection: 'strategy1' or 'strategy2'
STRATEGY = "strategy1"

# Input quadrant results CSV path
CSV_PATH = PROJECT_ROOT / "Plots/quandrantic_analysis/quadrant_results.csv"

# Output destination for the formatted CSV
OUTPUT_CSV = PROJECT_ROOT / f"Plots/quandrantic_analysis/consolidated_grid_{STRATEGY}_latex.csv"

# Formatting settings
AS_PERCENTAGE = True     # Multiply values and uncertainties by 100
MATH_DELIMITERS = True   # True: '$val \pm unc$', False: 'val \pm unc'
PRECISION = 2            # Number of decimal places (2 for percentages)
USE_LATEX_LABELS = True  # Use formatted ET and eta LaTeX range descriptions
INCLUDE_GLOBAL = True    # Append composite GLOBAL row at the bottom

print(f"Target strategy: {STRATEGY}")
print(f"Input CSV:        {CSV_PATH}")
print(f"Output CSV:       {OUTPUT_CSV}")
print(f"Percentage mode:  {AS_PERCENTAGE}")

### 2. Load Results & Inspect Strategy Name

In [ ]:
consolidator = QuadrantTableConsolidator(CSV_PATH)
strategy_name = consolidator.get_strategy_name(STRATEGY)

print(f"Configured Strategy Name: {strategy_name}")
print(f"Total input rows:         {len(consolidator.raw_data)}")

### 3. Generate 2D Grid Table ($E_T$ Energy Interval Blocks with SP, PF, PD Sub-Columns)

For each $E_T$ energy interval, the table contains three internal columns: `SP`, `PF`, and `PD` (scaled to percentages). If SP uncertainty is absent, only the metric value is formatted without propagation.

In [ ]:
grid_table = consolidator.build_et_grid_table(
    strategy=STRATEGY,
    as_percentage=AS_PERCENTAGE,
    math_delimiters=MATH_DELIMITERS,
    precision=PRECISION,
    use_latex_labels=USE_LATEX_LABELS,
    include_global=INCLUDE_GLOBAL,
)

display(grid_table)

### 4. Inspect Numeric Metrics (Optional Reference)

Raw numeric values and uncertainties (scaled to percentages) preserved as floating-point numbers.

In [ ]:
numeric_table = consolidator.build_numeric_table(
    strategy=STRATEGY,
    as_percentage=AS_PERCENTAGE,
    include_global=INCLUDE_GLOBAL,
)

display(numeric_table.head(10))

### 5. Export Formatted Table to CSV

In [ ]:
saved_path = consolidator.export_csv(grid_table, OUTPUT_CSV)
print(f"Consolidated grid table exported successfully to: {saved_path}")

### 6. LaTeX Code Generation

Ready-to-use LaTeX tabular snippet with multicolumn blocks for each $E_T$ energy interval, wrapped with `\resizebox{\textwidth}{!}{...}` to ensure proper document margin fit.

In [ ]:
caption = f"Regional performance metrics (SP, PF, PD in \\%) for {strategy_name} across $E_T$ and $|\\eta|$ intervals"
label = f"tab:regional_efficiencies_{STRATEGY}"

latex_snippet = consolidator.to_latex_snippet(
    grid_table,
    caption=caption,
    label=label,
    resize_box=True,
)

print(latex_snippet)